# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 1: Data Collection with SpaceX API
## IBM Applied Data Science Capstone

In [ ]:
!pip install requests pandas numpy
import requests
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Helper Functions - BoosterVersion, LaunchSite, Payload, Core Data

In [ ]:
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get('https://api.spacexdata.com/v4/rockets/' + str(x)).json()
            BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get('https://api.spacexdata.com/v4/launchpads/' + str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get('https://api.spacexdata.com/v4/payloads/' + load).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] != None:
            response = requests.get('https://api.spacexdata.com/v4/cores/' + core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## Task 1: Request and parse the SpaceX launch data

In [ ]:
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url)
print(response.status_code)
# Use json_normalize to convert json to dataframe
data = pd.json_normalize(response.json())
# Get the head of the dataframe
data.head()

In [ ]:
# Take subset - only features we want
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data.rename(columns={'date_utc': 'date'})
data['date'] = pd.to_datetime(data['date']).dt.date
data.head()

In [ ]:
# Call helper functions
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)
print(len(BoosterVersion), len(LaunchSite), len(PayloadMass), len(Outcome))

In [ ]:
launch_dict = {'FlightNumber': list(data['flight_number']), 'Date': list(data['date']), 'BoosterVersion': BoosterVersion, 'PayloadMass': PayloadMass, 'Orbit': Orbit, 'LaunchSite': LaunchSite, 'Outcome': Outcome, 'Flights': Flights, 'GridFins': GridFins, 'Reused': Reused, 'Legs': Legs, 'LandingPad': LandingPad, 'Block': Block, 'ReusedCount': ReusedCount, 'Serial': Serial, 'Longitude': Longitude, 'Latitude': Latitude}
# Create dataframe from launch_dict
data = pd.DataFrame(launch_dict)
data.head()

## Task 2: Filter dataframe to only include Falcon 9 launches

In [ ]:
# Filter Falcon 9 only
data_falcon9 = data[data['BoosterVersion'] != 'Falcon 1']
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9.head()

## Task 3: Dealing with Missing Values

In [ ]:
data_falcon9.isnull().sum()
# Calculate mean of PayloadMass
mean_payload = data_falcon9['PayloadMass'].mean()
print('Mean PayloadMass:', mean_payload)
# Replace NaN with mean
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, mean_payload)
data_falcon9.isnull().sum()
data_falcon9.to_csv('dataset_part_1.csv', index=False)
data_falcon9.head()

## Author - Nasri Prince - IBM Data Science Capstone